In [1]:
from google.colab import files
uploaded = files.upload()  # Tải lên: annonimized.csv và th-public.csv

Saving annonimized.csv to annonimized.csv


In [2]:
uploaded = files.upload()

Saving th-public.csv to th-public.csv


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [11]:
df = pd.read_csv("annonimized.csv")
print(df.columns.tolist())

["concat('it001',`assignment_id`)", "concat('it001',`problem_id`)", "concat('it001', username)", 'is_final', 'status', 'pre_score', 'coefficient', "concat('it001',`language_id`)", 'created_at', 'updated_at', 'judgement']


In [12]:
df = pd.read_csv("th-public.csv")
print(df.columns.tolist())

['hash', 'TH']


In [14]:
# STEP 2: Đọc dữ liệu
df = pd.read_csv("annonimized.csv")
scores = pd.read_csv("th-public.csv")

# Chuẩn hóa tên cột cho df
df.columns = [
    "assignment_id", "problem_id", "username", "is_final", "status",
    "pre_score", "coefficient", "language_id", "created_at",
    "updated_at", "judgement"
]

# Chuẩn hóa tên cột cho scores
scores.columns = [col.strip().lower() for col in scores.columns]
scores.columns = ["username", "score"]  # giả sử là cột hash và TH


In [15]:
# STEP 3: Trích xuất đặc trưng từ dữ liệu nộp bài
features = df.groupby("username").agg(
    n_submissions=('problem_id', 'count'),
    n_unique_problems=('problem_id', 'nunique'),
    n_final_submissions=('is_final', 'sum'),
    avg_pre_score=('pre_score', 'mean'),
    max_pre_score=('pre_score', 'max'),
    accepted_ratio=('status', lambda x: np.mean(x == 'Accepted')),
    avg_coefficient=('coefficient', 'mean'),
    language_diversity=('language_id', 'nunique'),
).reset_index()


In [18]:
# STEP 4: Ghép dữ liệu điểm
df_train = pd.merge(features, scores, on="username")

# Chuyển đổi cột 'score' sang dạng số và xử lý các giá trị không hợp lệ
df_train['score'] = pd.to_numeric(df_train['score'], errors='coerce')

# Loại bỏ các dòng có giá trị NaN trong cột 'score' sau khi chuyển đổi
df_train.dropna(subset=['score'], inplace=True)

# STEP 5: Huấn luyện mô hình
X = df_train.drop(columns=["username", "score"]).fillna(0)
y = df_train["score"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print("R² score trên tập validation:", r2_score(y_val, y_pred))

R² score trên tập validation: 0.2974409803718904


In [19]:
# Dự đoán và đánh giá
y_pred = model.predict(X_val)
print("R² score:", r2_score(y_val, y_pred))


R² score: 0.2974409803718904


In [24]:
# STEP 6: Dự đoán cho tất cả sinh viên trong annonimized.csv
all_usernames = df["username"].unique()
feature_full = pd.DataFrame({'username': all_usernames})
feature_full = feature_full.merge(features, on="username", how="left").fillna(0)

X_full = feature_full.drop(columns=["username"])
predicted_scores = model.predict(X_full)

# Làm tròn theo bước 0.5
rounded_scores = (predicted_scores * 2).round() / 2

# Tạo submission
submission = pd.DataFrame({
    "username": feature_full["username"],
    "TH": rounded_scores
})

# Xuất file nộp
submission.to_csv("submission.csv", index=False)

In [25]:
from google.colab import files
files.download("submission.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>